# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [15]:
# Only needed for Udacity workspace

import importlib.util
import sys
import uuid

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [16]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
from typing import List, Dict
import chromadb
from chromadb.utils import embedding_functions
import json
from tavily import TavilyClient
from lib.tooling  import tool

In [17]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [18]:
os.getenv("OPENAI_BASE_URL")

'https://openai.vocareum.com/v1'

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [19]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
@tool(
    name="retrieve_game",
    description="Retrieve relevant video game information from the local vector database based on a user query."
)
def retrieve_game(query: str) -> List[dict]:
    """
    Retrieve relevant video game information from the local vector database based on a user query.

    Args:
        query (str): A question about the game industry.

    Returns:
        List[dict]: A list of dictionaries containing game information.
    """
    print("==>Invoking tool: retrieve_game")
    chroma_client = chromadb.PersistentClient(path="chromadb")
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        api_base="https://openai.vocareum.com/v1"
    )

    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents']
    )
    # Extract relevant information from the results
    games = []
    for doc in results["documents"][0]:
        game_info = json.loads(doc)
        games.append(game_info)
   

    return games

retrieve_game("Nintendo")  # Example usage

==>Invoking tool: retrieve_game


[{'Name': 'Wii Sports',
  'Platform': 'Wii',
  'Genre': 'Sports',
  'Publisher': 'Nintendo',
  'Description': "A collection of sports games that utilize the Wii's motion controls, bundled with the console to showcase its capabilities.",
  'YearOfRelease': 2006},
 {'Name': 'Super Mario World',
  'Platform': 'Super Nintendo Entertainment System (SNES)',
  'Genre': 'Platformer',
  'Publisher': 'Nintendo',
  'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.',
  'YearOfRelease': 1990},
 {'Name': 'Super Mario 64',
  'Platform': 'Nintendo 64',
  'Genre': 'Platformer',
  'Publisher': 'Nintendo',
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'YearOfRelease': 1996}]

#### Evaluate Retrieval Tool

In [20]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool(
    name="evaluate_retrieval",
    description="Evaluate whether the retrieved documents contain sufficient and relevant information to answer the user's question accurately."
)
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> Dict:
    """
    Evaluate whether the retrieved documents contain sufficient and relevant information to answer the user's question accurately.

    Args:
        question (str): User question.
        retrieved_docs (List[str]): Retrieved documents from vector search.

    Returns:
        Dict: {
            "useful": bool,
            "description": str
        }
    """
    print("==>Invoking tool: evaluate_retrieval")
    print(str)
    print(retrieved_docs)
    llm = LLM(model="gpt-4o-mini", temperature=0)

    prompt = f"""
You are a retrieval evaluation agent.

Your task is to determine whether the retrieved documents contain enough relevant information
to answer the user's question accurately.

Evaluate:
1. Relevance of the retrieved documents
2. Coverage of the user's question
3. Missing information, if any
4. Whether the final answer can be generated confidently

Return ONLY valid JSON in the following format:

{{
    "useful": true,
    "description": "Detailed explanation"
}}

User Question:
{question}

Retrieved Documents:
{chr(10).join(retrieved_docs)}
"""

    response = llm.invoke(prompt)

    # Handle string response safely
    if isinstance(response, str):
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            return {
                "useful": False,
                "description": f"Failed to parse LLM response: {response}"
            }

    # If already parsed dict
    if isinstance(response, dict):
        return response

    return {
        "useful": False,
        "description": "Unexpected response format from LLM."
    }

#### Game Web Search Tool

In [21]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
@tool(
    name="game_web_search",
    description="Search the web for up-to-date video game and gaming industry information when local data is insufficient."
)
def game_web_search(question: str) -> Dict:
    """
    Search the web for up-to-date video game and gaming industry information when local data is insufficient.

    Args:
        question (str): User query about games or the gaming industry.

    Returns:
        Dict: {
            "answer": str,
            "results": List[dict]
        }
    """
    print("==>Invoking tool: game_web_search")
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    formatted_results = {
        "answer": response.get("answer", ""),
        "results": [
            {
                "title": result.get("title", ""),
                "url": result.get("url", ""),
                "content": result.get("content", "")
            }
            for result in response.get("results", [])
        ]
    }

    return formatted_results

### Agent

In [22]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
instructions = """
You are an AI Research Agent specialized in the video game industry.

Your responsibilities:
1. Answer user questions using your own knowledge whenever possible.
2. Use the `retrieve_game` tool to retrieve relevant game information from the local vector database.
3. Use the `evaluate_retrieval` tool to determine whether the retrieved information is sufficient to answer the user's question accurately.
4. If the retrieved information is insufficient or incomplete, use the `game_web_search` tool as a fallback source.
5. Combine information from available sources to produce a clear, factual, and helpful final response.

Guidelines:
- Prefer local database information when it is sufficient.
- Use web search only when necessary.
- Never fabricate information.
- Keep responses concise but informative.
- Clearly mention the source(s) used.

Final response format:

User Question:
<user question>

Final Answer:
<complete answer>

Sources:
- Local database
OR
- <full clickable URLs>

Tools Used:
- retrieve_game
- evaluate_retrieval
- game_web_search

If no tools were used, omit the "Tools Used" section.
"""


agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ],
    instructions=instructions
)

In [23]:
session_id = uuid.uuid4()
print(f"session_id = {session_id}")

session_id = 0a730b5d-4eb4-4fc6-aaf8-a6acf23882bb


In [24]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
result = agent.invoke("when is x box released?", session_id=session_id)
print(result.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
==>Invoking tool: retrieve_game
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
==>Invoking tool: evaluate_retrieval
<class 'str'>
['Halo Infinite', 'Kinect Adventures!', 'Minecraft']
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
==>Invoking tool: game_web_search
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
User Question:
when is x box released?

Final Answer:
The original Xbox was released on November 15, 2001. Subsequent consoles include the Xbox 360, which launched on November 22, 2005, and the Xbox One, released on November 22, 2013. The most recent console, the Xbox Series X and Series S, was launched on November 10, 2020. A new Xbox console is expected to be released around mid-to-late 

In [25]:
result = agent.invoke("who published Minecraft?", session_id=session_id)
print(result.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
==>Invoking tool: retrieve_game
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
User Question:
who published Minecraft?

Final Answer:
Minecraft was published by Mojang Studios.

Sources:
- Local database


In [26]:
result = agent.invoke("What is the a game for Nintendo Switch?", session_id=session_id)
print(result.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
==>Invoking tool: retrieve_game
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
User Question:
What is a game for Nintendo Switch?

Final Answer:
One popular game for the Nintendo Switch is "Mario Kart 8 Deluxe." It's a racing game that is an enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics. It was released in 2017.

Sources:
- Local database


In [27]:
result = agent.invoke("Summarize the last requests and generate consolidated report.", session_id=session_id)
print(result.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
**Consolidated Report of User Requests:**

1. **Request:** When is Xbox released?
   - **Response:** The original Xbox was released on November 15, 2001. Subsequent consoles include the Xbox 360 (November 22, 2005), Xbox One (November 22, 2013), and Xbox Series X/S (November 10, 2020). A new Xbox console is expected around mid-to-late 2026.
   - **Sources:** 
     - [IGN Article on Xbox Console Release Dates](https://www.ign.com/articles/all-xbox-console-release-dates-in-order)
     - [Wikipedia on Xbox Series X and Series S](https://en.wikipedia.org/wiki/Xbox_Series_X_and_Series_S)
     - [Asurion Article on Xbox History](https://www.asurion.com/connect/tech-tips/xbox-history-new-tt/)

2. **Request:** Who published Minecraft?
   - **Response:** Minecraft was published by Mojang Studios.
   - **Sources:** Local database


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes